In [1]:
import math
from pathlib import Path

def _lerp(p, q, t: float):
    return (p[0] + (q[0] - p[0]) * t, p[1] + (q[1] - p[1]) * t)

def _intersect_line_with_line(P, v, Q, w):
    """
    Solve P + s v = Q + u w for (s, u).
    Returns (s, u). Raises ValueError if parallel.
    """
    (px, py), (vx, vy) = P, v
    (qx, qy), (wx, wy) = Q, w

    # Solve:
    # px + s vx = qx + u wx
    # py + s vy = qy + u wy
    # => s*v - u*w = (q - p)
    det = vx * (-wy) - vy * (-wx)  # det([v, -w]) = -vx*wy + vy*wx
    if abs(det) < 1e-12:
        raise ValueError("Parallel lines; no unique intersection.")

    dx, dy = (qx - px), (qy - py)
    # Cramer's rule for s in matrix [ v  -w ]
    s = (dx * (-wy) - dy * (-wx)) / det
    u = (vx * dy - vy * dx) / det
    return s, u

def triangle_lattice_svg(n: int, unit: float = 40.0, margin: float = 12.0,
                         stroke: str = "#111", stroke_width: float = 2.0) -> str:
    """
    Draw an order-n triangular lattice inside a big equilateral triangle.
    Steps:
      1) draw big triangle (side length = n*unit)
      2) draw horizontal lines of lengths 1..n-1
      3) draw // direction lines
      4) draw \\ direction lines
    """
    if n <= 0:
        raise ValueError("n must be positive")

    L = n * unit
    H = L * math.sqrt(3) / 2.0

    # Big triangle vertices (apex up)
    A = (margin + L / 2.0, margin)          # apex
    B = (margin, margin + H)                # left base
    C = (margin + L, margin + H)            # right base

    svg_w = L + 2 * margin
    svg_h = H + 2 * margin

    def line(p, q):
        return (f'<line x1="{p[0]:.3f}" y1="{p[1]:.3f}" '
                f'x2="{q[0]:.3f}" y2="{q[1]:.3f}" '
                f'stroke="{stroke}" stroke-width="{stroke_width}" />')

    elems = []

    # 1) Big triangle border
    elems.append(line(A, B))
    elems.append(line(A, C))
    elems.append(line(B, C))

    # 2) Horizontal lines (parallel to BC): for i = 1..n-1
    # Endpoints are points on AB and AC at the same fraction i/n from A.
    for i in range(1, n):
        t = i / n
        p = _lerp(A, B, t)  # on AB
        q = _lerp(A, C, t)  # on AC
        elems.append(line(p, q))

    # Direction vectors of the triangle sides
    vAB = (B[0] - A[0], B[1] - A[1])
    vAC = (C[0] - A[0], C[1] - A[1])
    vBC = (C[0] - B[0], C[1] - B[1])

    # 3) // direction lines: lines parallel to AB.
    # Take point on AC at t=i/n, draw line through it parallel to AB until it hits BC.
    for i in range(1, n):
        t = i / n
        P = _lerp(A, C, t)  # on AC
        # Intersect: P + s*vAB with BC: B + u*vBC
        s, u = _intersect_line_with_line(P, vAB, B, vBC)
        Q = (B[0] + u * vBC[0], B[1] + u * vBC[1])
        elems.append(line(P, Q))

    # 4) \\ direction lines: lines parallel to AC.
    # Take point on AB at t=i/n, draw line through it parallel to AC until it hits BC.
    for i in range(1, n):
        t = i / n
        P = _lerp(A, B, t)  # on AB
        # Intersect: P + s*vAC with BC: B + u*vBC
        s, u = _intersect_line_with_line(P, vAC, B, vBC)
        Q = (B[0] + u * vBC[0], B[1] + u * vBC[1])
        elems.append(line(P, Q))

    svg = f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" width="{svg_w:.0f}" height="{svg_h:.0f}" viewBox="0 0 {svg_w:.3f} {svg_h:.3f}">
  {'\n  '.join(elems)}
</svg>
"""
    return svg

def save_triangle_lattice_svg(n: int, out_path: str | None = None, **kwargs) -> Path:
    svg = triangle_lattice_svg(n, **kwargs)
    if out_path is None:
        out_path = f"tri_lattice_n{n}.svg"
    p = Path(out_path)
    p.write_text(svg, encoding="utf-8")
    return p

if __name__ == "__main__":
    p = save_triangle_lattice_svg(8, unit=40, stroke_width=2)
    print("Saved:", p)

Saved: tri_lattice_n8.svg


In [3]:
import math
from pathlib import Path
from typing import List, Tuple, Optional

Point = Tuple[float, float]

def cross(ax, ay, bx, by) -> float:
    return ax * by - ay * bx

def clip_line_to_convex_polygon(P: Point, v: Point, poly: List[Point]) -> Optional[Tuple[Point, Point]]:
    """
    Clip infinite line X(t)=P + t*v against a convex polygon (CCW).
    Returns endpoints (X(t0), X(t1)) or None if no intersection segment.
    Uses half-plane constraints: for each edge Vi->Vj, inside is left side.
    """
    px, py = P
    vx, vy = v

    t_lo = -1e30
    t_hi =  1e30

    m = len(poly)
    for i in range(m):
        x1, y1 = poly[i]
        x2, y2 = poly[(i + 1) % m]
        ex, ey = (x2 - x1, y2 - y1)

        # Condition: cross(e, X - Vi) >= 0
        # => cross(e, (P - Vi) + t*v) >= 0
        c0 = cross(ex, ey, px - x1, py - y1)
        c1 = cross(ex, ey, vx, vy)

        if abs(c1) < 1e-12:
            # Line parallel to this edge's supporting line; either always inside or always outside
            if c0 < 0:
                return None
            continue

        t_bound = -c0 / c1
        if c1 > 0:
            # t >= t_bound
            if t_bound > t_lo:
                t_lo = t_bound
        else:
            # c1 < 0 => t <= t_bound
            if t_bound < t_hi:
                t_hi = t_bound

        if t_lo > t_hi:
            return None

    A = (px + t_lo * vx, py + t_lo * vy)
    B = (px + t_hi * vx, py + t_hi * vy)
    return A, B

def hex_lattice_svg(n: int, unit: float = 40.0, margin: float = 12.0,
                    stroke: str = "#111", stroke_width: float = 2.0,
                    outer_stroke_width: float = 3.0) -> str:
    """
    Draw a side-length-n hexagon on a triangular lattice, then draw:
      - horizontal family (b = 0..2n): 2n+1 lines
      - // family (a = -n..n): 2n+1 lines
      - \\ family (a+b = 0..2n): 2n+1 lines
    Order: outer hex -> horizontal -> // -> \\ .
    """
    if n <= 0:
        raise ValueError("n must be positive")

    h = unit * math.sqrt(3) / 2.0
    u = (unit, 0.0)
    v = (unit / 2.0, h)
    w = (v[0] - u[0], v[1] - u[1])  # (-unit/2, h)

    def to_xy(a: float, b: float) -> Point:
        # a*u + b*v
        return (a * u[0] + b * v[0], a * u[1] + b * v[1])

    # Hex vertices in (a,b) coordinates (CCW), side length n:
    # (0,0)->(n,0)->(n,n)->(0,2n)->(-n,2n)->(-n,n)
    verts_ab = [(0, 0), (n, 0), (n, n), (0, 2*n), (-n, 2*n), (-n, n)]
    poly = [to_xy(a, b) for a, b in verts_ab]

    # Compute bbox to translate into positive with margin
    xs = [p[0] for p in poly]
    ys = [p[1] for p in poly]
    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)

    tx = margin - minx
    ty = margin - miny

    def T(p: Point) -> Point:
        return (p[0] + tx, p[1] + ty)

    polyT = [T(p) for p in poly]

    svg_w = (maxx - minx) + 2 * margin
    svg_h = (maxy - miny) + 2 * margin

    def line(p: Point, q: Point, sw: float) -> str:
        return (f'<line x1="{p[0]:.3f}" y1="{p[1]:.3f}" '
                f'x2="{q[0]:.3f}" y2="{q[1]:.3f}" '
                f'stroke="{stroke}" stroke-width="{sw}" />')

    elems = []

    # 1) Outer hex border
    for i in range(len(polyT)):
        elems.append(line(polyT[i], polyT[(i+1) % len(polyT)], outer_stroke_width))

    # Helper to clip & append a line in original (untranslated) then translate endpoints
    def add_family_line(P0: Point, dirv: Point, sw: float):
        seg = clip_line_to_convex_polygon(P0, dirv, poly)
        if seg is None:
            return
        A, B = seg
        elems.append(line(T(A), T(B), sw))

    # 2) Horizontal family: b = k for k=0..2n, line direction u
    # Pick point at a=0: P = (0,k)
    for k in range(0, 2*n + 1):
        P0 = to_xy(0.0, float(k))
        add_family_line(P0, u, stroke_width)

    # 3) // family: a = k for k=-n..n, line direction v
    # Pick point at b=0: P = (k,0)
    for k in range(-n, n + 1):
        P0 = to_xy(float(k), 0.0)
        add_family_line(P0, v, stroke_width)

    # 4) \\ family: a+b = k for k=0..2n, line direction w
    # Param: P = (k,0) + t*w  (i.e., start at (a=k,b=0))
    for k in range(0, 2*n + 1):
        P0 = to_xy(float(k), 0.0)
        add_family_line(P0, w, stroke_width)

    svg = f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{svg_w:.0f}" height="{svg_h:.0f}"
     viewBox="0 0 {svg_w:.3f} {svg_h:.3f}">
  {'\n  '.join(elems)}
</svg>
"""
    return svg

def save_hex_lattice_svg(n: int, out_path: str | None = None, **kwargs) -> Path:
    svg = hex_lattice_svg(n, **kwargs)
    if out_path is None:
        out_path = f"hex_lattice_n{n}.svg"
    p = Path(out_path)
    p.write_text(svg, encoding="utf-8")
    return p

if __name__ == "__main__":
    p = save_hex_lattice_svg(3, unit=40, stroke_width=2, outer_stroke_width=3, margin=12)
    print("Saved:", p)

Saved: hex_lattice_n3.svg


In [5]:
import math
from pathlib import Path
from typing import List, Tuple, Optional

Point = Tuple[float, float]

def _cross(ax, ay, bx, by) -> float:
    return ax * by - ay * bx

def clip_line_to_convex_polygon(P: Point, v: Point, poly_ccw: List[Point]) -> Optional[Tuple[Point, Point]]:
    px, py = P
    vx, vy = v

    t_lo = -1e30
    t_hi =  1e30

    m = len(poly_ccw)
    for i in range(m):
        x1, y1 = poly_ccw[i]
        x2, y2 = poly_ccw[(i + 1) % m]
        ex, ey = (x2 - x1, y2 - y1)

        # inside: cross(e, X - Vi) >= 0
        c0 = _cross(ex, ey, px - x1, py - y1)
        c1 = _cross(ex, ey, vx, vy)

        if abs(c1) < 1e-12:
            if c0 < 0:
                return None
            continue

        t_bound = -c0 / c1
        if c1 > 0:
            t_lo = max(t_lo, t_bound)
        else:
            t_hi = min(t_hi, t_bound)

        if t_lo > t_hi:
            return None

    A = (px + t_lo * vx, py + t_lo * vy)
    B = (px + t_hi * vx, py + t_hi * vy)
    return A, B

def _dot(a: Point, b: Point) -> float:
    return a[0]*b[0] + a[1]*b[1]

def _sub(a: Point, b: Point) -> Point:
    return (a[0]-b[0], a[1]-b[1])

def segment_length_units_along_dir(A: Point, B: Point, d: Point, unit: float) -> int:
    """
    격자선 방향 d(가로, //, \\)에 대한 '격자 단위 길이'를 계산.
    유클리드 길이 대신, 방향으로의 투영을 사용해서 정수 길이가 안정적으로 나옴.
    """
    v = _sub(B, A)
    dd = _dot(d, d)
    if dd < 1e-12:
        return 0
    proj = abs(_dot(v, d)) / math.sqrt(dd)   # length along direction in XY
    return int(round(proj / unit))

def david_star_lattice_svg(
    n: int,
    unit: float = 40.0,
    margin: float = 12.0,
    stroke: str = "#111",
    stroke_width: float = 2.0,
    outer_stroke_width: float = 3.0,
    use_length_filter: bool = False,   # ✅ 기본은 끔 (일단 무조건 그려지게)
) -> str:
    if n <= 0:
        raise ValueError("n must be positive")

    L = 3 * n * unit
    H = L * math.sqrt(3) / 2.0

    # Upright triangle
    A = (L / 2.0, 0.0)
    B = (0.0, H)
    C = (L, H)
    tri_up = [A, C, B]  # CCW

    # Centroid
    G = ((A[0] + B[0] + C[0]) / 3.0, (A[1] + B[1] + C[1]) / 3.0)

    # Inverted triangle by 180° rotation about centroid
    def rot180(p: Point) -> Point:
        return (2 * G[0] - p[0], 2 * G[1] - p[1])

    tri_dn = [rot180(p) for p in tri_up]

    # Lattice directions
    h = unit * math.sqrt(3) / 2.0
    dir_h = (unit, 0.0)         # horizontal
    dir_slash = (unit/2.0, h)   # //
    dir_back  = (-unit/2.0, h)  # \\

    # Bounding box for both triangles
    all_pts = tri_up + tri_dn
    minx = min(p[0] for p in all_pts)
    maxx = max(p[0] for p in all_pts)
    miny = min(p[1] for p in all_pts)
    maxy = max(p[1] for p in all_pts)

    tx = margin - minx
    ty = margin - miny

    def T(p: Point) -> Point:
        return (p[0] + tx, p[1] + ty)

    svg_w = (maxx - minx) + 2 * margin
    svg_h = (maxy - miny) + 2 * margin

    def line(p: Point, q: Point, sw: float) -> str:
        return (f'<line x1="{p[0]:.3f}" y1="{p[1]:.3f}" '
                f'x2="{q[0]:.3f}" y2="{q[1]:.3f}" '
                f'stroke="{stroke}" stroke-width="{sw}" />')

    elems: List[str] = []

    # Borders
    def add_border(poly: List[Point], sw: float):
        for i in range(len(poly)):
            elems.append(line(T(poly[i]), T(poly[(i+1) % len(poly)]), sw))

    add_border(tri_up, outer_stroke_width)
    add_border(tri_dn, outer_stroke_width)

    # length groups (원래 의도: 짧은 군 + 긴 군)
    short_set = set(range(1, n))            # 1..n-1
    long_set  = set(range(2*n, 3*n))        # 2n..3n-1

    def add_line_family(P0: Point, dv: Point, dv_for_len: Point):
        segs = []
        s1 = clip_line_to_convex_polygon(P0, dv, tri_up)
        if s1 is not None: segs.append(s1)
        s2 = clip_line_to_convex_polygon(P0, dv, tri_dn)
        if s2 is not None: segs.append(s2)

        for (a, b) in segs:
            if use_length_filter:
                lu = segment_length_units_along_dir(a, b, dv_for_len, unit)
                if (lu not in short_set) and (lu not in long_set):
                    continue
            elems.append(line(T(a), T(b), stroke_width))

    # Sweep enough parallel lines
    sweep = 6 * n  # 넉넉히

    # Horizontal: y = G_y + k*h
    for k in range(-sweep, sweep + 1):
        y = G[1] + k * h
        P0 = (G[0] - 10 * L, y)   # far left anchor
        add_line_family(P0, dir_h, dir_h)

    # // : anchors shift along \\ to get parallel offsets
    for k in range(-sweep, sweep + 1):
        P0 = (G[0] + k * dir_back[0], G[1] + k * dir_back[1])
        add_line_family(P0, dir_slash, dir_slash)

    # \\ : anchors shift along // to get parallel offsets
    for k in range(-sweep, sweep + 1):
        P0 = (G[0] + k * dir_slash[0], G[1] + k * dir_slash[1])
        add_line_family(P0, dir_back, dir_back)

    svg = f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{svg_w:.0f}" height="{svg_h:.0f}"
     viewBox="0 0 {svg_w:.3f} {svg_h:.3f}">
  {'\n  '.join(elems)}
</svg>
"""
    return svg

def save_david_star_lattice_svg(n: int, out_path: str | None = None, **kwargs) -> Path:
    svg = david_star_lattice_svg(n, **kwargs)
    if out_path is None:
        out_path = f"david_star_lattice_n{n}.svg"
    p = Path(out_path)
    p.write_text(svg, encoding="utf-8")
    return p

if __name__ == "__main__":
    # ✅ 먼저 이걸로: 내부 줄이 반드시 보임
    p = save_david_star_lattice_svg(3, unit=40, use_length_filter=False)
    print("Saved:", p)


Saved: david_star_lattice_n3.svg


In [7]:
from pathlib import Path

def square_grid_svg(
    n: int,
    unit: float = 40.0,
    margin: float = 12.0,
    stroke: str = "#111",
    stroke_width: float = 2.0,
    outer_stroke_width: float = 3.0,
) -> str:
    """
    Draw an n x n square grid (n^2 cells) as SVG.
    Order:
      1) outer square
      2) horizontal inner lines at y = i*unit, i=1..n-1
      3) vertical inner lines at x = i*unit, i=1..n-1
    """
    if n <= 0:
        raise ValueError("n must be positive")

    L = n * unit
    x0, y0 = margin, margin
    x1, y1 = margin + L, margin + L

    svg_w = L + 2 * margin
    svg_h = L + 2 * margin

    def line(xa, ya, xb, yb, sw):
        return (f'<line x1="{xa:.3f}" y1="{ya:.3f}" '
                f'x2="{xb:.3f}" y2="{yb:.3f}" '
                f'stroke="{stroke}" stroke-width="{sw}" />')

    elems = []

    # 1) outer square
    elems.append(line(x0, y0, x1, y0, outer_stroke_width))
    elems.append(line(x1, y0, x1, y1, outer_stroke_width))
    elems.append(line(x1, y1, x0, y1, outer_stroke_width))
    elems.append(line(x0, y1, x0, y0, outer_stroke_width))

    # 2) horizontal inner lines
    for i in range(1, n):
        y = y0 + i * unit
        elems.append(line(x0, y, x1, y, stroke_width))

    # 3) vertical inner lines
    for i in range(1, n):
        x = x0 + i * unit
        elems.append(line(x, y0, x, y1, stroke_width))

    svg = f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{svg_w:.0f}" height="{svg_h:.0f}"
     viewBox="0 0 {svg_w:.3f} {svg_h:.3f}">
  {'\n  '.join(elems)}
</svg>
"""
    return svg

def save_square_grid_svg(n: int, out_path: str | None = None, **kwargs) -> Path:
    svg = square_grid_svg(n, **kwargs)
    if out_path is None:
        out_path = f"square_grid_n{n}.svg"
    p = Path(out_path)
    p.write_text(svg, encoding="utf-8")
    return p

if __name__ == "__main__":
    p = save_square_grid_svg(10, unit=30, margin=12, stroke_width=1.5, outer_stroke_width=3)
    print("Saved:", p)

Saved: square_grid_n10.svg
